In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import requests
import pandas as pd
import numpy as np
from sqlalchemy.engine import create_engine
from sqlalchemy.engine.url import URL
from iacele import OdooAPI

load_dotenv(dotenv_path="../.env.local")
load_dotenv(dotenv_path="../.env")

# .env.local apunta a la API local (SQLite). Esta libreta escribe en Cloud SQL
# a través de la API de Cloud Run, que es la que sí llega a PostgreSQL.
url = "https://prueba01-backend-169247078761.us-west2.run.app"
print("API de PostgreSQL:", url)

api = OdooAPI()

ruta_carpeta = Path.home().joinpath('Documents/Inventario/A1 2025')


Las altas van a la API de Cloud Run. Desde esta PC el puerto 5432 de Cloud SQL no responde, así que el engine de abajo solo sirve para leer cuando la red lo permita.


In [ ]:
DATABASE_URL = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("GCLOUD_SQL_USER"),
    password=os.getenv("GCLOUD_SQL_PASSWORD"),
    host=os.getenv("GCLOUD_SQL_IP"),
    port=5432,
    database=os.getenv("GCLOUD_SQL_DB"),
    query={"gssencmode": "disable", "sslnegotiation": "postgres"},
)
engine = create_engine(DATABASE_URL, pool_pre_ping=True)


Ingresar empleados que no están en Odoo

In [ ]:
usuarios = pd.read_csv(Path("empleados_a_usuarios.csv"))
usuarios["id_odoo"] = usuarios["id_odoo"].astype(int)
usuarios["empleado_id"] = usuarios["empleado_id"].astype(int)
id_app = dict(zip(usuarios["id_odoo"], usuarios["empleado_id"]))

manuales = {
    1: {
        "job_title": "Director General",
        "department": "Dirección General",
        "warehouse": "Cabo San Lucas",
        "hire_date_str": "2013-01-01",
    },
    2: {
        "job_title": "Administración General",
        "department": "Dirección General",
        "warehouse": "Cabo San Lucas",
        "hire_date_str": "2026-08-22",
    },
}

ids_por_odoo = {
    int(empleado["id_odoo"]): int(empleado["id"])
    for empleado in requests.get(f"{url}/employees/").json()
}

def alta_empleado(body):
    id_odoo = int(body["id_odoo"])
    if id_odoo in ids_por_odoo:
        print("ya está en PostgreSQL:", id_odoo, body["name"], "id", ids_por_odoo[id_odoo])
        return ids_por_odoo[id_odoo]
    x = requests.post(f"{url}/employees/new", json=body)
    if not x.ok:
        print(x, " para ", id_odoo)
        print(x.json())
        return None
    creado = x.json()
    ids_por_odoo[id_odoo] = int(creado["id"])
    pedido = body.get("id")
    if pedido is not None and int(creado["id"]) != int(pedido):
        print("PostgreSQL asignó id", creado["id"], "en lugar de", int(pedido), "para", body["name"])
    return int(creado["id"])

ingresados = []
for _, persona in usuarios.loc[usuarios["id_odoo"].isin(manuales)].iterrows():
    id_odoo = int(persona["id_odoo"])
    body = {
        "id": int(persona["empleado_id"]),
        "id_odoo": id_odoo,
        "name": persona["nombre_completo"],
        **manuales[id_odoo],
    }
    if alta_empleado(body) is not None:
        ingresados.append(id_odoo)

print(f"{len(ingresados)} de {len(manuales)} empleados fuera de Odoo listos en PostgreSQL")


Ingresar empleados desde Odoo (tiempo_real)

In [ ]:
payload = {
        "domain": [("active", "!=", False)],
        "fields": [
            "id",
            "name",
            "job_title",
            "department_id",
            "x_hire_date",
            "x_warehouse_id",
        ],
        "context": {"lang": "es_MX"},
    }

empleados_data = api.search_read("hr.employee", payload=payload)
empleados_df = pd.DataFrame(empleados_data)
empleados_df.rename(columns={
    "id": "id_odoo",
    "x_hire_date": "hire_date_str",
}, inplace=True)

empleados_df["department"] = empleados_df["department_id"].apply(lambda x: "Ventas Piso" if "Ventas Piso" in x[1] else x[1])
empleados_df["warehouse"] = empleados_df["x_warehouse_id"].apply(lambda x: "Cabo San Lucas" if x[0]==2 else "San José Del Cabo")
empleados_df.drop(columns=["department_id", "x_warehouse_id"], inplace=True)

empleados_df = empleados_df.loc[empleados_df["id_odoo"].isin(id_app)].copy()
empleados_df = empleados_df.loc[~empleados_df["id_odoo"].isin(manuales)].copy()

en_odoo = set(empleados_df["id_odoo"].astype(int))
faltan_en_odoo = sorted(set(id_app) - set(manuales) - en_odoo)
if faltan_en_odoo:
    print("En el CSV pero no en Odoo:", faltan_en_odoo)

ingresados = []
id_errores = []

for _, row in empleados_df.iterrows():
    hire = row["hire_date_str"]
    if not isinstance(hire, str) or not hire:
        print("sin fecha de ingreso:", int(row["id_odoo"]), row["name"])
        id_errores.append(int(row["id_odoo"]))
        continue
    body = {
        "id": int(id_app[int(row["id_odoo"])]),
        "id_odoo": int(row["id_odoo"]),
        "name": row["name"],
        "job_title": row["job_title"] if isinstance(row["job_title"], str) else "",
        "department": row["department"] if isinstance(row["department"], str) else "",
        "warehouse": row["warehouse"] if isinstance(row["warehouse"], str) else "",
        "hire_date_str": hire[:10],
    }
    if alta_empleado(body) is not None:
        ingresados.append(body["id_odoo"])
    else:
        id_errores.append(body["id_odoo"])

print(f"{len(ingresados)} de {len(empleados_df)} empleados de Odoo listos en PostgreSQL")


Promover a usuario de la app con el username del CSV. La contraseña inicial es 123456.


In [ ]:
ids_en_postgres = {
    int(empleado["id_odoo"]): int(empleado["id"])
    for empleado in requests.get(f"{url}/employees/").json()
}

desfasados = []
for _, persona in usuarios.iterrows():
    id_odoo = int(persona["id_odoo"])
    esperado = int(persona["empleado_id"])
    actual = ids_en_postgres.get(id_odoo)
    if actual != esperado:
        desfasados.append((persona["nombre_completo"], actual, esperado))

if desfasados:
    print("No se promueve nadie. El id de PostgreSQL no coincide con el CSV:")
    for nombre, actual, esperado in desfasados:
        print(f"  {nombre}: postgres {actual}, csv {esperado}")
else:
    promovidos = []
    for _, persona in usuarios.iterrows():
        empleado_id = int(persona["empleado_id"])
        username = str(persona["username"]).strip()
        body = {"is_app_user": True, "username": username}
        x = requests.patch(f"{url}/employees/authentication/{empleado_id}/promote", json=body)
        if x.ok or x.json().get("detail") == "El id del empleado ya tiene usuario":
            promovidos.append(username)
        else:
            print(x, " para ", username)
            print(x.json())
    print(f"{len(promovidos)} de {len(usuarios)} empleados con usuario. Contraseña inicial: 123456")


Ingresar equipos desde archivo

In [ ]:
eq_oficial_df = pd.read_excel("C:/Users/lic_ferre/Documents/Inventario/equipos 2026 A1.xlsx")

endpoint = url + '/teams/new'

ingresados = []
id_errores = []

for i, row in eq_oficial_df.iterrows():
    producto = row.fillna(None).to_dict()
    x = requests.post(endpoint, json = producto)
    if x.ok:
        ingresados.append(x.ok)
    else:
        print(x, ' para ', row['Eq'])
        print(x.json())
        id_errores.append(row.name)

print(f'{len(ingresados)} de {len(eq_oficial_df)} equipos ingresados con éxito')

Ingresar productos desde archivo lp_oficial

In [ ]:
lp_oficial_df = pd.read_excel("C:/Users/lic_ferre/Documents/Inventario/Lista de Productos 2026 A1.xlsx")
lp_oficial_df['code'] = lp_oficial_df['code'].astype(str)

endpoint = url + '/products/new'

ingresados = []
id_errores = []

for i, row in lp_oficial_df.iterrows():
    producto = row.fillna(None).to_dict()
    x = requests.post(endpoint, json = producto)
    if x.ok:
        ingresados.append(x.ok)
    else:
        print(x, ' para ', row['id_odoo'])
        print(x.json())
        id_errores.append(row.name)

print(f'{len(ingresados)} de {len(lp_oficial_df)} productos ingresados con éxito')